# CogniGuard walkthrough video (free Colab)

Renders a narrated MP4: runs the real Stage 3 eval to get the true numbers, generates the
**Microsoft Andrew** voice, builds slides, and stitches everything with ffmpeg. No AVX needed.

**Runtime -> Run all.** Push your repo first so the clone includes `evaluation/` and `video/`.

In [ ]:
!pip -q install edge-tts sentence-transformers nest_asyncio
import os
if not os.path.isdir('cogniguard'):
    !git clone -q https://github.com/louisawamuyu/cogniguard
%cd /content/cogniguard
!git pull -q || true

In [ ]:
# 1) measure Stage 3 for real -> true numbers for the narration
!python evaluation/eval_stage3.py | tail -15
import json
r = json.load(open('evaluation/stage3_eval_results.json'))
catch = round(r['operating']['recall']*100)
fpr = round(r['operating']['fpr']*100)
print('MEASURED: catch', catch, '% | false-positive', fpr, '%')

In [ ]:
SEGMENTS = [
 ("What is CogniGuard?", "CogniGuard is a safety tool for AI agents. An agent reads messages, calls tools, and takes actions for you. CogniGuard's job is to keep an eye on that worker."),
 ("It began as a flight recorder", "It started as a flight recorder. Every prompt, every tool call, every file, and what it cost, all recorded, so you could replay a run and see what happened. Great for audits. But a recorder only tells you what went wrong after the fact. It watches. It does not step in."),
 ("The gap", "Attackers don't always use the obvious words. One writes, ignore all previous instructions. Another writes, please disregard what you were told. Same attack, different words. I wanted CogniGuard to catch the second one, live, before the agent acts."),
 ("What it is now: a real-time gate", "So CogniGuard grew a second job. A real-time gate that checks a message before the agent acts. It works in stages, cheapest first. Stage one is fast pattern matching for known attack phrases. Most messages are settled right there."),
 ("Stage three: meaning, not words", "The interesting part is stage three, the semantic stage. It turns a message into numbers that capture its meaning, then compares that meaning to a library of known attacks. Please disregard what you were told lands close to ignore all previous instructions, even with almost no shared words. If the meaning is close enough, CogniGuard flags it."),
 ("How we know: an honest test", "A claim like that only counts if it is measured. So I built a test, with attacks in fresh wording the system had never seen, plus ordinary safe messages, including tricky ones that contain words like ignore or password. I measured two things. How often it catches the rephrased attacks, and how often it wrongly flags a safe message."),
 ("The result", f"Here is the honest result. On the held-out test, stage three caught {catch} percent of rephrased attacks, with a {fpr} percent false alarm rate on safe messages."),
 ("Where this is going", "This is early, honest work. One small test set, one model, and the next step is a bigger, harder set. But the shape is real. CogniGuard has gone from a recorder that explains the past, to a gate that can act in the present. And every number here was measured, not asserted."),
]
print(len(SEGMENTS), 'segments; result segment =', SEGMENTS[6][1])

In [ ]:
import os, asyncio, subprocess, textwrap, nest_asyncio, edge_tts
from PIL import Image, ImageDraw, ImageFont
nest_asyncio.apply()
os.makedirs('video/out', exist_ok=True)
VOICE='en-US-AndrewNeural'
FONT='/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
FB  ='/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'
def font(p,s): return ImageFont.truetype(p,s)
def slide(path, heading, body):
    W,H=1280,720; img=Image.new('RGB',(W,H),(15,23,42)); d=ImageDraw.Draw(img)
    d.text((70,55), heading, font=font(FB,50), fill=(56,189,248))
    y=185
    for line in textwrap.wrap(body, 52):
        d.text((70,y), line, font=font(FONT,32), fill=(226,232,240)); y+=52
    d.text((70,H-55),'CogniGuard', font=font(FB,26), fill=(100,116,139))
    img.save(path)
async def tts(text, path): await edge_tts.Communicate(text, VOICE).save(path)
clips=[]
for i,(h,b) in enumerate(SEGMENTS):
    mp3,png,mp4=f'video/out/s{i}.mp3',f'video/out/s{i}.png',f'video/out/s{i}.mp4'
    asyncio.get_event_loop().run_until_complete(tts(b, mp3))
    slide(png,h,b)
    subprocess.run(['ffmpeg','-y','-loop','1','-i',png,'-i',mp3,'-c:v','libx264','-tune','stillimage','-c:a','aac','-b:a','192k','-pix_fmt','yuv420p','-shortest',mp4],check=True,capture_output=True)
    clips.append(f's{i}.mp4'); print('rendered segment', i)
open('video/out/list.txt','w').write(''.join(f"file '{c}'\n" for c in clips))
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i','video/out/list.txt','-c','copy','video/out/cogniguard_walkthrough.mp4'],check=True,capture_output=True)
print('DONE -> video/out/cogniguard_walkthrough.mp4')

In [ ]:
from google.colab import files
files.download('video/out/cogniguard_walkthrough.mp4')